# Imports

In [152]:
# data processing
import numpy as np
import pandas as pd 

# machine learning
from keras.models import Sequential
from keras.layers import Dense
from scikeras.wrappers import KerasClassifier, KerasRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

# utils
import time
from datetime import timedelta


In [153]:
# Input files
file_train='../data/train.csv'
file_test='../data/test.csv'

In [154]:
seed = 69
np.random.seed(seed)

In [155]:
train_df = pd.read_csv(file_train,index_col='PassengerId')

In [156]:
# Show the columns
train_df.columns.values

array(['Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'Embarked'], dtype=object)

In [157]:
# Show the shape
train_df.shape

(891, 11)

In [158]:
# preview the training dara
train_df.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [159]:
# Show that there is NaN data (Age,Fare Embarked), that needs to be handled during data cleansing
train_df.isnull().sum()

Survived      0
Pclass        0
Name          0
Sex           0
Age         177
SibSp         0
Parch         0
Ticket        0
Fare          0
Cabin       687
Embarked      2
dtype: int64

In [160]:
train_df.Fare.sort_values()

PassengerId
816      0.0000
807      0.0000
414      0.0000
482      0.0000
303      0.0000
         ...   
439    263.0000
342    263.0000
738    512.3292
680    512.3292
259    512.3292
Name: Fare, Length: 891, dtype: float64

A function for simple data cleansing <br>
- Drop unwanted features ['Name', 'Ticket', 'Cabin']<br>
- Fill missing data: Age and Fare with the mean, Embarked with most frequent value<br>
- Convert categorical features into numeric<br>
- Convert Embarked to one-hot<br>

In [161]:
def prep_data(df):
    # Drop unwanted features
    df = df.drop(['Name', 'Ticket', 'Cabin'], axis=1)
    
    # Fill missing data: Age and Fare with the mean, Embarked with most frequent value
    df[['Age']] = df[['Age']].fillna(value=df[['Age']].mean())
    df[['Fare']] = df[['Fare']].fillna(value=df[['Fare']].mean())
    df[['Embarked']] = df[['Embarked']].fillna(value=df['Embarked'].value_counts().idxmax())
    
    # Convert categorical  features into numeric
    df['Sex'] = df['Sex'].map( {'female': 1, 'male': 0} ).astype(int)
      
    # Convert Embarked to one-hot
    enbarked_one_hot = pd.get_dummies(df['Embarked'], prefix='Embarked')
    df = df.drop('Embarked', axis=1)
    df = df.join(enbarked_one_hot)

    return df

In [162]:
train_df = prep_data(train_df)
train_df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age           0
SibSp         0
Parch         0
Fare          0
Embarked_C    0
Embarked_Q    0
Embarked_S    0
dtype: int64

In [163]:
# X contains all columns except 'Survived'  
X = train_df.drop(['Survived'], axis=1).values.astype(float)

# It is almost always a good idea to perform some scaling of input values when using neural network models (jb).

scale = StandardScaler()
X = scale.fit_transform(X)

# Y is just the 'Survived' column
Y = train_df['Survived'].values

Simple Network using Keras <br>
- Input layer with 16 neuron (units/outputs).<br>
- Two hidden layers.<br>
- Output layer with a single neuron and sigmoid activation function to output a value between 0 and 1.<br>
- Optimizer and init will be searched with GridSearch.<br>

In [164]:
verbose=2 # Use in classifier

In [165]:
def create_model(optimizer='adam', init='uniform') -> Sequential:
    # create model
    if verbose: print("**Create model with optimizer: %s; init: %s" % (optimizer, init) )
    model = Sequential()
    model.add(Dense(16, input_dim=X.shape[1], kernel_initializer=init, activation='relu'))
    model.add(Dense(8, kernel_initializer=init, activation='relu'))
    model.add(Dense(4, kernel_initializer=init, activation='relu'))
    model.add(Dense(1, kernel_initializer=init, activation='sigmoid'))
    # Compile model
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model

In [166]:
run_gridsearch = False

if run_gridsearch:
    
    start_time = time.time()
    if verbose: print (time.strftime( "%H:%M:%S " + "GridSearch started ... " ) )
    optimizers = ['rmsprop', 'adam']
    inits = ['glorot_uniform', 'normal', 'uniform']
    epochs = [50, 100, 200, 400]
    batches = [5, 10, 20]
    
    model = KerasClassifier(build_fn=create_model, verbose=verbose)
    
    param_grid = dict(optimizer=optimizers, epochs=epochs, batch_size=batches, init=inits)
    grid = GridSearchCV(estimator=model, param_grid=param_grid)
    grid_result = grid.fit(X, Y)
    
    # summarize results
    print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
    means = grid_result.cv_results_['mean_test_score']
    stds = grid_result.cv_results_['std_test_score']
    params = grid_result.cv_results_['params']
    if verbose: 
        for mean, stdev, param in zip(means, stds, params):
            print("%f (%f) with: %r" % (mean, stdev, param))
        elapsed_time = time.time() - start_time  
        print ("Time elapsed: ",timedelta(seconds=elapsed_time))
        
    best_epochs = grid_result.best_params_['epochs']
    best_batch_size = grid_result.best_params_['batch_size']
    best_init = grid_result.best_params_['init']
    best_optimizer = grid_result.best_params_['optimizer']
    
else:
    # pre-selected paramters
    best_epochs = 200
    best_batch_size = 5
    best_init = 'glorot_uniform'
    best_optimizer = 'rmsprop'

In [167]:
# Create a classifier with best parameters
model_pred = KerasClassifier(build_fn=create_model, optimizer=best_optimizer, init=best_init, epochs=best_epochs, batch_size=best_batch_size, verbose=verbose)
model_pred.fit(X, Y)

# Read test data
test_df = pd.read_csv(file_test,index_col='PassengerId')
# Prep and clean data
test_df = prep_data(test_df)
# Create X_test
X_test = test_df.values.astype(float)
# Scaling
X_test = scale.transform(X_test)

# Predict 'Survived'
prediction = model_pred.predict(X_test)

**Create model with optimizer: adam; init: glorot_uniform
Epoch 1/200


c:\Users\alexandre.karneff\Documents\GitHub\alexandre.karneff\envWorkshop\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\alexandre.karneff\Documents\GitHub\alexandre.karneff\envWorkshop\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


179/179 - 1s - 5ms/step - accuracy: 0.6184 - loss: 0.6165
Epoch 2/200
179/179 - 0s - 911us/step - accuracy: 0.7374 - loss: 0.5383
Epoch 3/200
179/179 - 0s - 981us/step - accuracy: 0.8227 - loss: 0.5081
Epoch 4/200
179/179 - 0s - 969us/step - accuracy: 0.8193 - loss: 0.4834
Epoch 5/200
179/179 - 0s - 979us/step - accuracy: 0.8193 - loss: 0.4603
Epoch 6/200
179/179 - 0s - 928us/step - accuracy: 0.8193 - loss: 0.4329
Epoch 7/200
179/179 - 0s - 976us/step - accuracy: 0.8272 - loss: 0.4182
Epoch 8/200
179/179 - 0s - 923us/step - accuracy: 0.8328 - loss: 0.4080
Epoch 9/200
179/179 - 0s - 938us/step - accuracy: 0.8316 - loss: 0.4041
Epoch 10/200
179/179 - 0s - 907us/step - accuracy: 0.8316 - loss: 0.4001
Epoch 11/200
179/179 - 0s - 914us/step - accuracy: 0.8339 - loss: 0.3957
Epoch 12/200
179/179 - 0s - 912us/step - accuracy: 0.8361 - loss: 0.3945
Epoch 13/200
179/179 - 0s - 952us/step - accuracy: 0.8406 - loss: 0.3912
Epoch 14/200
179/179 - 0s - 995us/step - accuracy: 0.8406 - loss: 0.3872
E

In [168]:
submission = pd.DataFrame({
    'PassengerId': test_df.index,
    'Survived': prediction
})

submission.sort_values('PassengerId', inplace=True)    
submission.to_csv('submission-simple-cleansing.csv', index=False)

In [169]:
print(prediction.shape)

(418,)


# Part 2 - Predict the missing data in the Age feature

In [170]:
# Read the data
file_train='../data/train.csv'
file_test='../data/test.csv'
    
df_train = pd.read_csv(file_train,index_col='PassengerId')
df_test = pd.read_csv(file_test,index_col='PassengerId')  
l = len(df_train.index)
    

## All data train and test in one dataframe 
dfa = pd.concat([df_train, df_test])

# Drop unwanted features
dfa = dfa.drop(['Name', 'Ticket', 'Cabin'], axis=1)
    
# Fill missing data: Fare with mean, Embarked with most frequent value
dfa[['Fare']] = dfa[['Fare']].fillna(value=dfa[['Fare']].mean())
dfa[['Embarked']] = dfa[['Embarked']].fillna(value=dfa['Embarked'].value_counts().idxmax())
    
# Convert categorical features into numeric
dfa['Sex'] = dfa['Sex'].map( {'female': 1, 'male': 0} ).astype(int)

# Convert 'Embarked' to one-hot
enbarked_one_hot = pd.get_dummies(dfa['Embarked'], prefix='Embarked')
dfa = dfa.drop('Embarked', axis=1)
dfa = dfa.join(enbarked_one_hot)

In [171]:
dfa.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S
PassengerId,,,,,,,,,,
1,0.0,3,0,22.0,1,0,7.2500,False,False,True
2,1.0,1,1,38.0,1,0,71.2833,True,False,False
3,1.0,3,1,26.0,0,0,7.9250,False,False,True
4,1.0,1,1,35.0,1,0,53.1000,False,False,True
5,0.0,3,0,35.0,0,0,8.0500,False,False,True


In [172]:
# Split data in to training set (Age not null) and 'to-be-predicted' set (Age in nan)
df_age_train = dfa[dfa.Age.notnull()]
df_age_nan = dfa[dfa.Age.isnull()]

Predict age <br>
- Create input X, output Y and X_test<br>
- The model for the regression. The regression problem may have a single output neuron and the neuron may have no activation function (jb).<br>
- Create a pipeline, do cross-validation and predict<br>
- Generate new input files for the training and test data but with predicted age<br>

In [173]:
# split data into input X and output Y
X = df_age_train.drop(['Age', 'Survived'], axis=1).values.astype(float)
Y = df_age_train['Age'].values.astype(float)

X_test = df_age_nan.drop(['Age', 'Survived'], axis=1).values.astype(float)

def age_model():
    # create model
    model = Sequential()
    model.add(Dense(32, input_dim=X.shape[1], kernel_initializer='normal', activation='relu'))
    model.add(Dense(16, kernel_initializer='normal', activation='relu'))
    model.add(Dense(8, kernel_initializer='normal', activation='relu'))
    model.add(Dense(1, kernel_initializer='normal'))
    # Compile model
    model.compile(loss='mean_squared_error', optimizer='adam')
    return model

In [174]:
estimators = []
estimators.append(('standardize', StandardScaler()))
estimators.append(('mlp', KerasRegressor(build_fn=age_model, epochs=100, batch_size=5, verbose=verbose)))
pipeline = Pipeline(estimators)

In [175]:
kfold = KFold(n_splits=2)
results = cross_val_score(pipeline, X, Y, cv=kfold)
print("Result: %.2f (%.2f) MSE" % (results.mean(), results.std()))

Epoch 1/100


c:\Users\alexandre.karneff\Documents\GitHub\alexandre.karneff\envWorkshop\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\alexandre.karneff\Documents\GitHub\alexandre.karneff\envWorkshop\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


105/105 - 1s - 6ms/step - loss: 1085.1030
Epoch 2/100
105/105 - 0s - 926us/step - loss: 784.6974
Epoch 3/100
105/105 - 0s - 931us/step - loss: 187.0743
Epoch 4/100
105/105 - 0s - 917us/step - loss: 159.6093
Epoch 5/100
105/105 - 0s - 1ms/step - loss: 156.0351
Epoch 6/100
105/105 - 0s - 916us/step - loss: 153.1585
Epoch 7/100
105/105 - 0s - 925us/step - loss: 151.3333
Epoch 8/100
105/105 - 0s - 926us/step - loss: 150.2538
Epoch 9/100
105/105 - 0s - 942us/step - loss: 149.2234
Epoch 10/100
105/105 - 0s - 920us/step - loss: 148.6994
Epoch 11/100
105/105 - 0s - 959us/step - loss: 148.8624
Epoch 12/100
105/105 - 0s - 916us/step - loss: 147.0875
Epoch 13/100
105/105 - 0s - 943us/step - loss: 147.0017
Epoch 14/100
105/105 - 0s - 923us/step - loss: 146.8462
Epoch 15/100
105/105 - 0s - 930us/step - loss: 145.5003
Epoch 16/100
105/105 - 0s - 911us/step - loss: 144.9095
Epoch 17/100
105/105 - 0s - 935us/step - loss: 144.7384
Epoch 18/100
105/105 - 0s - 922us/step - loss: 143.9784
Epoch 19/100
105

c:\Users\alexandre.karneff\Documents\GitHub\alexandre.karneff\envWorkshop\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\alexandre.karneff\Documents\GitHub\alexandre.karneff\envWorkshop\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


105/105 - 1s - 6ms/step - loss: 1091.1373
Epoch 2/100
105/105 - 0s - 986us/step - loss: 564.9609
Epoch 3/100
105/105 - 0s - 964us/step - loss: 169.3603
Epoch 4/100
105/105 - 0s - 954us/step - loss: 156.3379
Epoch 5/100
105/105 - 0s - 936us/step - loss: 152.7558
Epoch 6/100
105/105 - 0s - 940us/step - loss: 150.4652
Epoch 7/100
105/105 - 0s - 945us/step - loss: 149.7189
Epoch 8/100
105/105 - 0s - 943us/step - loss: 148.2903
Epoch 9/100
105/105 - 0s - 930us/step - loss: 147.3830
Epoch 10/100
105/105 - 0s - 922us/step - loss: 147.2942
Epoch 11/100
105/105 - 0s - 934us/step - loss: 145.7395
Epoch 12/100
105/105 - 0s - 917us/step - loss: 146.2767
Epoch 13/100
105/105 - 0s - 937us/step - loss: 144.7742
Epoch 14/100
105/105 - 0s - 936us/step - loss: 145.3355
Epoch 15/100
105/105 - 0s - 946us/step - loss: 144.4118
Epoch 16/100
105/105 - 0s - 1ms/step - loss: 144.7543
Epoch 17/100
105/105 - 0s - 950us/step - loss: 143.9144
Epoch 18/100
105/105 - 0s - 918us/step - loss: 142.8355
Epoch 19/100
105

In [176]:
pipeline.fit(X, Y)
prediction_train = pipeline.predict(X)
prediction_test = pipeline.predict(X_test)

Epoch 1/100


c:\Users\alexandre.karneff\Documents\GitHub\alexandre.karneff\envWorkshop\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\alexandre.karneff\Documents\GitHub\alexandre.karneff\envWorkshop\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


210/210 - 1s - 4ms/step - loss: 718.6436
Epoch 2/100
210/210 - 0s - 821us/step - loss: 163.9879
Epoch 3/100
210/210 - 0s - 793us/step - loss: 156.7833
Epoch 4/100
210/210 - 0s - 802us/step - loss: 154.4097
Epoch 5/100
210/210 - 0s - 819us/step - loss: 153.0219
Epoch 6/100
210/210 - 0s - 799us/step - loss: 150.4883
Epoch 7/100
210/210 - 0s - 786us/step - loss: 149.2840
Epoch 8/100
210/210 - 0s - 812us/step - loss: 149.4263
Epoch 9/100
210/210 - 0s - 803us/step - loss: 147.2325
Epoch 10/100
210/210 - 0s - 814us/step - loss: 147.5950
Epoch 11/100
210/210 - 0s - 808us/step - loss: 146.9802
Epoch 12/100
210/210 - 0s - 802us/step - loss: 146.3658
Epoch 13/100
210/210 - 0s - 807us/step - loss: 146.4108
Epoch 14/100
210/210 - 0s - 806us/step - loss: 145.7020
Epoch 15/100
210/210 - 0s - 849us/step - loss: 145.2987
Epoch 16/100
210/210 - 0s - 834us/step - loss: 145.2878
Epoch 17/100
210/210 - 0s - 810us/step - loss: 146.0471
Epoch 18/100
210/210 - 0s - 790us/step - loss: 144.5098
Epoch 19/100
21

Generate new input files for the training and test data with predicted age

In [177]:
# Create a data frame with PassengerId and predicted age
df_age_pred = pd.DataFrame({
    'PassengerId': df_age_nan.index,
    'Age_pred': prediction_test.astype(int)
})
df_age_pred.set_index('PassengerId', inplace=True)
   

# Add column with predicted age to the dataframe with all data (dfa)

dfa2 = pd.concat([df_train, df_test])
dfa_pred = pd.concat([dfa2, df_age_pred], axis=1)   

# Update Age column with prediction where nan and remove Age_pred
dfa_pred['Age'] = np.where(pd.isnull(dfa_pred['Age']), dfa_pred['Age_pred'] , dfa_pred['Age'])
dfa_pred = dfa_pred.drop(['Age_pred'], axis=1)

# Create new files
l = len(df_train)
df_train2 = dfa_pred[0:l] 
df_test2 = dfa_pred[l:] 
df_test2 = df_test2.drop(['Survived'], axis=1)

df_train2.to_csv('train-age-predicted.csv')
df_test2.to_csv('test-age-predicted.csv')

# Part 3 - Wrangle, prepare, cleanse the titanic data manually

Re-read data, drop the Cabin and Ticket features

In [178]:
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')
train_df = train_df.drop(['Ticket', 'Cabin'], axis=1)
test_df = test_df.drop(['Ticket', 'Cabin'], axis=1)
combine = [train_df, test_df]

Creating new feature extracting from existing ...

In [179]:
for dataset in combine:
    dataset['Title'] = dataset.Name.str.extract(' ([A-Za-z]+)\.', expand=False)

for dataset in combine:
    dataset['Title'] = dataset['Title'].replace(['Lady', 'Countess','Capt', 'Col',\
 	'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')

    dataset['Title'] = dataset['Title'].replace('Mlle', 'Miss')
    dataset['Title'] = dataset['Title'].replace('Ms', 'Miss')
    dataset['Title'] = dataset['Title'].replace('Mme', 'Mrs')
    
train_df[['Title', 'Survived']].groupby(['Title'], as_index=False).mean()

<>:2: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
C:\Users\alexandre.karneff\AppData\Local\Temp\ipykernel_18152\1402529983.py:2: SyntaxWarning: invalid escape sequence '\.'
  dataset['Title'] = dataset.Name.str.extract(' ([A-Za-z]+)\.', expand=False)


,Title,Survived
0,Master,0.575000
1,Miss,0.702703
2,Mr,0.156673
3,Mrs,0.793651
4,Rare,0.347826


In [180]:
train_df.Title.value_counts()

Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64

In [181]:
# convert the categorical titles to ordinal.
title_mapping = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5}
for dataset in combine:
    dataset['Title'] = dataset['Title'].map(title_mapping)
    dataset['Title'] = dataset['Title'].fillna(0)

# Drop Name feature
train_df = train_df.drop(['Name'], axis=1)
test_df = test_df.drop(['Name'], axis=1)
combine = [train_df, test_df]

In [182]:
for dataset in combine:
    dataset['Sex'] = dataset['Sex'].map( {'female': 1, 'male': 0} ).astype(int)

In [183]:
train_df.head(10)

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title
0,1,0,3,0,22.0,1,0,7.2500,S,1
1,2,1,1,1,38.0,1,0,71.2833,C,3
2,3,1,3,1,26.0,0,0,7.9250,S,2
3,4,1,1,1,35.0,1,0,53.1000,S,3
4,5,0,3,0,35.0,0,0,8.0500,S,1
5,6,0,3,0,NaN,0,0,8.4583,Q,1
6,7,0,1,0,54.0,0,0,51.8625,S,1
7,8,0,3,0,2.0,3,1,21.0750,S,4
8,9,1,3,1,27.0,0,2,11.1333,S,3
9,10,1,2,1,14.0,1,0,30.0708,C,3


In [184]:
guess_ages = np.zeros((2,3))
for dataset in combine:
    for i in range(0, 2):
        for j in range(0, 3):
            guess_df = dataset[(dataset['Sex'] == i) & \
                                  (dataset['Pclass'] == j+1)]['Age'].dropna()

            age_guess = guess_df.median()

            # Convert random age float to nearest .5 age
            guess_ages[i,j] = int( age_guess/0.5 + 0.5 ) * 0.5
            
    for i in range(0, 2):
        for j in range(0, 3):
            dataset.loc[ (dataset.Age.isnull()) & (dataset.Sex == i) & (dataset.Pclass == j+1),\
                    'Age'] = guess_ages[i,j]

    dataset['Age'] = dataset['Age'].astype(int)

In [185]:
for dataset in combine:    
    dataset.loc[ dataset['Age'] <= 16, 'Age'] = 0
    dataset.loc[(dataset['Age'] > 16) & (dataset['Age'] <= 32), 'Age'] = 1
    dataset.loc[(dataset['Age'] > 32) & (dataset['Age'] <= 48), 'Age'] = 2
    dataset.loc[(dataset['Age'] > 48) & (dataset['Age'] <= 64), 'Age'] = 3
    dataset.loc[ dataset['Age'] > 64, 'Age'] = 4

train_df.head(10)

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title
0,1,0,3,0,1,1,0,7.2500,S,1
1,2,1,1,1,2,1,0,71.2833,C,3
2,3,1,3,1,1,0,0,7.9250,S,2
3,4,1,1,1,2,1,0,53.1000,S,3
4,5,0,3,0,2,0,0,8.0500,S,1
5,6,0,3,0,1,0,0,8.4583,Q,1
6,7,0,1,0,3,0,0,51.8625,S,1
7,8,0,3,0,0,3,1,21.0750,S,4
8,9,1,3,1,1,0,2,11.1333,S,3
9,10,1,2,1,0,1,0,30.0708,C,3


In [186]:
train_df.Age.value_counts()

Age
1    495
2    216
0    100
3     69
4     11
Name: count, dtype: int64

Create new feature for FamilySize which combines Parch and SibSp

In [187]:
combine = [train_df, test_df]
for dataset in combine:
    dataset['FamilySize'] = dataset['SibSp'] + dataset['Parch'] + 1

train_df[['FamilySize', 'Survived']].groupby(['FamilySize'], as_index=False).mean().sort_values(by='Survived', ascending=False)

,FamilySize,Survived
3,4,0.724138
2,3,0.578431
1,2,0.552795
6,7,0.333333
0,1,0.303538
4,5,0.200000
5,6,0.136364
7,8,0.000000
8,11,0.000000


In [188]:
for dataset in combine:
    dataset['IsAlone'] = 0
    dataset.loc[dataset['FamilySize'] == 1, 'IsAlone'] = 1

train_df[['IsAlone', 'Survived']].groupby(['IsAlone'], as_index=False).mean()

,IsAlone,Survived
0,0,0.505650
1,1,0.303538


Drop Parch, SibSp, and FamilySize features in favor of IsAlone

In [189]:
train_df = train_df.drop(['Parch', 'SibSp', 'FamilySize'], axis=1)
test_df = test_df.drop(['Parch', 'SibSp', 'FamilySize'], axis=1)
combine = [train_df, test_df]

Create an artificial feature combining Pclass and Age

In [190]:
for dataset in combine:
    dataset['Age*Class'] = dataset.Age * dataset.Pclass

train_df.loc[:, ['Age*Class', 'Age', 'Pclass']].head(10)

,Age*Class,Age,Pclass
0,3,1,3
1,2,2,1
2,3,1,3
3,2,2,1
4,6,2,3
5,3,1,3
6,3,3,1
7,0,0,3
8,3,1,3
9,0,0,2


Completing a categorical feature

In [191]:
freq_port = train_df.Embarked.dropna().mode()[0]

for dataset in combine:
    dataset['Embarked'] = dataset['Embarked'].fillna(freq_port)
    
for dataset in combine:
    dataset['Embarked'] = dataset['Embarked'].map( {'S': 0, 'C': 1, 'Q': 2} ).astype(int)

test_df['Fare'].fillna(test_df['Fare'].dropna().median(), inplace=True)

C:\Users\alexandre.karneff\AppData\Local\Temp\ipykernel_18152\1559063093.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_df['Fare'].fillna(test_df['Fare'].dropna().median(), inplace=True)


Fare feature to ordinal values based on the FareBand

In [192]:
for dataset in combine:
    dataset.loc[ dataset['Fare'] <= 7.91, 'Fare'] = 0
    dataset.loc[(dataset['Fare'] > 7.91) & (dataset['Fare'] <= 14.454), 'Fare'] = 1
    dataset.loc[(dataset['Fare'] > 14.454) & (dataset['Fare'] <= 31), 'Fare']   = 2
    dataset.loc[ dataset['Fare'] > 31, 'Fare'] = 3
    dataset['Fare'] = dataset['Fare'].astype(int)

combine = [train_df, test_df]

Save wrangled training and test data to files

In [193]:
train_df.to_csv('../data/train-wrangled.csv',index=False)
test_df.to_csv('../data/test-wrangled.csv',index=False)

In [194]:
# Input files
file_train='../data/train-wrangled.csv'
file_test='../data/test-wrangled.csv'

# read training data
train_df = pd.read_csv(file_train,index_col='PassengerId')

In [195]:
train_df.columns.values

array(['Survived', 'Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Title',
       'IsAlone', 'Age*Class'], dtype=object)

In [196]:
# Show the shape
train_df.shape

(891, 9)

In [197]:
# preview the training dara
train_df.head(20)

,Survived,Pclass,Sex,Age,Fare,Embarked,Title,IsAlone,Age*Class
PassengerId,,,,,,,,,
1,0,3,0,1,0,0,1,0,3
2,1,1,1,2,3,1,3,0,2
3,1,3,1,1,1,0,2,1,3
4,1,1,1,2,3,0,3,0,2
5,0,3,0,2,1,0,1,1,6
6,0,3,0,1,1,2,1,1,3
7,0,1,0,3,3,0,1,1,3
8,0,3,0,0,2,0,4,0,0
9,1,3,1,1,1,0,3,0,3


In [198]:
# Show that there isn't any NaN data 
train_df.isnull().sum()

Survived     0
Pclass       0
Sex          0
Age          0
Fare         0
Embarked     0
Title        0
IsAlone      0
Age*Class    0
dtype: int64

In [199]:
# X contains all columns except 'Survived'  
X = train_df.drop(['Survived'], axis=1).values.astype(float)

# It is almost always a good idea to perform some scaling of input values when using neural network models (jb).

scale = StandardScaler()
X = scale.fit_transform(X)

# Y is just the 'Survived' column
Y = train_df['Survived'].values

In [200]:
train_df.Age.value_counts()

Age
1    495
2    216
0    100
3     69
4     11
Name: count, dtype: int64

In [201]:
run_gridsearch = False

if run_gridsearch:
    
    start_time = time.time()
    if verbose: print (time.strftime( "%H:%M:%S " + "GridSearch started ... " ) )
    optimizers = ['rmsprop', 'adam']
    inits = ['glorot_uniform', 'normal', 'uniform']
    epochs = [50, 100, 200, 400]
    batches = [5, 10, 20]
    
    model = KerasClassifier(build_fn=create_model, verbose=verbose)
    
    param_grid = dict(optimizer=optimizers, epochs=epochs, batch_size=batches, init=inits)
    grid = GridSearchCV(estimator=model, param_grid=param_grid)
    grid_result = grid.fit(X, Y)
    
    # summarize results
    print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
    means = grid_result.cv_results_['mean_test_score']
    stds = grid_result.cv_results_['std_test_score']
    params = grid_result.cv_results_['params']
    if verbose: 
        for mean, stdev, param in zip(means, stds, params):
            print("%f (%f) with: %r" % (mean, stdev, param))
        elapsed_time = time.time() - start_time  
        print ("Time elapsed: ",timedelta(seconds=elapsed_time))
        
    best_epochs = grid_result.best_params_['epochs']
    best_batch_size = grid_result.best_params_['batch_size']
    best_init = grid_result.best_params_['init']
    best_optimizer = grid_result.best_params_['optimizer']
    
else:
    # pre-selected paramters
    best_epochs = 200
    best_batch_size = 5
    best_init = 'glorot_uniform'
    best_optimizer = 'rmsprop'

In [202]:
# Create a classifier with best parameters
model_pred = KerasClassifier(build_fn=create_model, optimizer=best_optimizer, init=best_init, epochs=best_epochs, batch_size=best_batch_size, verbose=verbose)
model_pred.fit(X, Y)

# Read test data
test_df = pd.read_csv(file_test,index_col='PassengerId')

# Create X_test
X_test = test_df.values.astype(float)
# Scaling
X_test = scale.transform(X_test)

# Predict 'Survived'
prediction = model_pred.predict(X_test)

**Create model with optimizer: adam; init: glorot_uniform
Epoch 1/200


c:\Users\alexandre.karneff\Documents\GitHub\alexandre.karneff\envWorkshop\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\alexandre.karneff\Documents\GitHub\alexandre.karneff\envWorkshop\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


179/179 - 1s - 4ms/step - accuracy: 0.6869 - loss: 0.6195
Epoch 2/200
179/179 - 0s - 854us/step - accuracy: 0.7688 - loss: 0.5183
Epoch 3/200
179/179 - 0s - 964us/step - accuracy: 0.8114 - loss: 0.4726
Epoch 4/200
179/179 - 0s - 865us/step - accuracy: 0.8171 - loss: 0.4457
Epoch 5/200
179/179 - 0s - 877us/step - accuracy: 0.8215 - loss: 0.4270
Epoch 6/200
179/179 - 0s - 863us/step - accuracy: 0.8204 - loss: 0.4172
Epoch 7/200
179/179 - 0s - 875us/step - accuracy: 0.8283 - loss: 0.4088
Epoch 8/200
179/179 - 0s - 848us/step - accuracy: 0.8350 - loss: 0.4048
Epoch 9/200
179/179 - 0s - 875us/step - accuracy: 0.8373 - loss: 0.4022
Epoch 10/200
179/179 - 0s - 858us/step - accuracy: 0.8294 - loss: 0.3990
Epoch 11/200
179/179 - 0s - 892us/step - accuracy: 0.8339 - loss: 0.3975
Epoch 12/200
179/179 - 0s - 843us/step - accuracy: 0.8328 - loss: 0.3973
Epoch 13/200
179/179 - 0s - 874us/step - accuracy: 0.8316 - loss: 0.3952
Epoch 14/200
179/179 - 0s - 866us/step - accuracy: 0.8305 - loss: 0.3930
E